In [ ]:
import scanpy as sc
import pandas as pd
from Xenium_5K_IPF_constants import WORK_PATH, CELL_TYPE_3M, SUBTYPE_HATCHES

In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/integrated_anndata.h5ad")
ecnp_ad = sc.read_h5ad(f"{WORK_PATH}AnnData/EcNP.h5ad")
celltype_df = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
cp_df = pd.read_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index_col="cell_id")

In [ ]:
hoge = ["cell_type_major", "subtype"]
obs_new = pd.merge(adata.obs, celltype_df[hoge], left_index=True, right_index=True, how="left")
for h,c in zip(hoge, [CELL_TYPE_3M, SUBTYPE_HATCHES]):
    obs_new[h] = obs_new[h].fillna("Not assigned")
    obs_new[h] = pd.Categorical(obs_new[h], categories=list(c)+["Not assigned"], ordered=True)

In [ ]:
obs_new_2 = pd.merge(obs_new, ecnp_ad.obs[["EcNP"]], left_index=True, right_index=True, how="left")
obs_new_2["EcNP"]=obs_new_2["EcNP"].cat.add_categories(["Not epithelial"])
obs_new_2["EcNP"]=obs_new_2["EcNP"].fillna("Not epithelial")

In [ ]:
adata.obs = pd.merge(obs_new_2, cp_df[["C_position"]], left_index=True, right_index=True, how="left")

In [ ]:
adata.write(f"{WORK_PATH}AnnData/upload_data.h5ad")